# RNA-seq → `expr.h5` (BulkRNABert, Colab **T4 GPU**)

Embeds every tumour RNA-seq file of the team's 873 patients with the authors' recipe
(GDC `tpm_unstranded`, `bulk_rna_bert_gtex_encode`, layer 4, mean over genes).
At the end you attach `expr.h5` (about 1 MB) in the chat; the Parquet export, the contract check
and the commit happen there.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Run one cell at a time and
compare with **Expected**. If Colab disconnects, run cell 1 again, then continue from the cell you
were on; downloads and embeddings resume.

**1. Setup.** Expected last lines: `TOTAL 873 306 567 ...` and
`clinical rows: 884 | study patients with a fold: 873`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess, zipfile
PRIVATE = '/content/drive/MyDrive/DL4H_private'
STAGE = f'{PRIVATE}/tcga10'                      # survives disconnects
WORK = '/content/work'                           # local scratch
AUTH = '/content/multimodal-cancer-modeling'     # authors' public repo
TOOLS = '/content/tcga-multimodal-embeddings'
TOOLS_URL = 'https://github.com/YaseenM2/tcga-multimodal-embeddings.git'
for d in (STAGE, WORK):
    os.makedirs(d, exist_ok=True)
if not os.path.isdir(AUTH):
    subprocess.run(['git', 'clone', '-q', '--depth', '1',
                    'https://github.com/StevenSong/multimodal-cancer-modeling.git', AUTH], check=True)
if not os.path.isdir(TOOLS):
    if subprocess.run(['git', 'clone', '-q', TOOLS_URL, TOOLS]).returncode != 0:
        # Repo not published yet: upload tcga-multimodal-embeddings.zip to /content first.
        zipfile.ZipFile('/content/tcga-multimodal-embeddings.zip').extractall('/content')
%cd {TOOLS}
!pip -q install -r requirements.txt
!python scripts/select_cohort.py --clinical {AUTH}/data/clinical.csv --splits {AUTH}/results/split_cases.csv --out-dir {WORK} | tail -3
CASES = f'{WORK}/split_cases.csv'                # the team's 873 case_ids

**2. Find each patient's RNA-seq files** via the public GDC API (no login).
Expected: JSON with `kept_cases: 873`, `kept_files` around 900, `total_gib` around 3.5, and
`study_cases_without_file: 0`. **Paste this output in the chat.**

In [ ]:
!python scripts/prepare_expression_manifest.py \
    --expr-manifest {AUTH}/data/expr-manifest.txt --splits {CASES} \
    --out-map {STAGE}/expr_files_tcga10.csv --out-manifest {STAGE}/expr-manifest-tcga10.txt

**3. Download them** (about 3.5 GB, MD5-checked; rerun to resume).
Expected last line: `ALL EXPRESSION FILES PRESENT AND VERIFIED!`.

In [ ]:
!python scripts/download_expression.py --map {STAGE}/expr_files_tcga10.csv --out-dir /content/expr_raw

**4. Install BulkRNABert** (JAX for CUDA, Haiku, the authors' fork at a pinned commit; the fork
includes the 24 MB checkpoint). Expected: the `jax` package list shows `jax-cuda12-*` and no `jax-cuda13-*`; three lines
`check N: exit=0 [CudaDevice(id=0)] clash=False`; `18 passed ...`; a list with a `CudaDevice`.

In [ ]:
!pip -q install -r requirements-expression.txt
![ -d /content/BulkRNABert ] || git clone -q https://github.com/StevenSong/BulkRNABert.git /content/BulkRNABert
!git -C /content/BulkRNABert checkout -q c85162ff18913313d0bdea4c3c1bc6d00f69b193
!pip -q install --no-deps -e /content/BulkRNABert
# Colab ships a newer CUDA 13 JAX plugin. With ours installed both try to register, and if
# the CUDA 13 one wins the process aborts (PJRT API mismatch). Remove it completely.
import glob
!pip uninstall -y jax-cuda13-plugin jax-cuda13-pjrt 2>&1 | grep -v -i 'not installed'
for d in (glob.glob('/usr/local/lib/python3*/dist-packages/jax_plugins/xla_cuda13*')
          + glob.glob('/usr/local/lib/python3*/dist-packages/jax_cuda13*')):
    print('removing leftover', d); shutil.rmtree(d, ignore_errors=True)
!pip list 2>/dev/null | grep -i -E '^jax'
for i in range(3):  # plugin order varies per process, so check a few fresh ones
    r = subprocess.run(['python', '-c', 'import jax; print(jax.devices())'],
                       capture_output=True, text=True)
    print(f'check {i + 1}: exit={r.returncode} {r.stdout.strip()} clash={"ALREADY_EXISTS" in r.stderr}')
!python -m pytest -q -p no:cacheprovider tests/test_bulkrnabert_chunked.py tests/test_embed_expr_bulkrnabert.py 2>&1 | tail -1
!python -c "import jax; print(jax.devices())"

**5. Pilot: 5 files.** Expected: `embedded 1 files (... s/file)` lines and `new: 5`. The first file
includes compilation time. **Paste this output in the chat** (it tells us how long step 6 takes).

In [ ]:
EMBED = (f'python scripts/embed_expr_bulkrnabert.py --expr-dir /content/expr_raw '
         f'--gene-list {AUTH}/data/bulkrnabert_gene_list.txt '
         f'--weights-folder /content/BulkRNABert/checkpoints --output-h5 /content/expr.h5')
H5_DRIVE = f'{STAGE}/expr.h5'
if os.path.exists(H5_DRIVE) and not os.path.exists('/content/expr.h5'):
    shutil.copy(H5_DRIVE, '/content/expr.h5')    # resume after a disconnect
!{EMBED} --max-new 5

**6. Everything else,** in batches of 100, copying `expr.h5` to Drive after each batch.
After a disconnect: cell 1, then 3 (fast re-check/re-download), 4, 5 (it resumes and adds 5), then this one.
Expected final line: `EXPRESSION EMBEDDING COMPLETE!`.

In [ ]:
import shlex
while True:
    run = subprocess.run(shlex.split(EMBED) + ['--max-new', '100'], capture_output=True, text=True)
    if run.returncode != 0:
        print(run.stdout[-1500:], run.stderr[-3000:]); break
    shutil.copy('/content/expr.h5', H5_DRIVE)
    print(' | '.join(l for l in run.stdout.splitlines() if l.split(':')[0] in ('new', 'skipped', 'remaining')))
    if 'EMBEDDING COMPLETE' in run.stdout:
        print('EXPRESSION EMBEDDING COMPLETE!'); break

**7. Check and hand over.** Expected: `EXPRESSION AUDIT PASSED!` and a Parquet summary with
`"cases_written": 873`. **Paste the output, then attach `expr.h5` in the chat**: in Google Drive it
is `My Drive/DL4H_private/tcga10/expr.h5` (or use the download link this cell prints).

In [ ]:
import shlex
shutil.copy('/content/expr.h5', H5_DRIVE)
check = subprocess.run(shlex.split(EMBED) + ['--check-only'], capture_output=True, text=True)
print(check.stdout.strip())
if check.returncode == 0:
    !python scripts/export_parquet.py --modality expr --h5 /content/expr.h5 --cases {CASES} --output {STAGE}/expr.parquet
    from google.colab import files
    files.download('/content/expr.h5')
else:
    print('Not finished yet: run cell 6 (it resumes), then this cell again.')